##Creating a table of A+E healthcare use linked to maternity interpreter cohort 

Purpose - to link A+E attendance in year preconception with maternity interpreter cohort

Dependencies - this notebook requires CCU063_03-D01, CCU063_03-D02 and CCU063_03-D03 to have saved their output tables in order to run

Authors - Majel McGranahan supported by Lars Murdock

Reviewed - Not reviewed

#0 Parameters

In [0]:
%run "./CCU063_03-D01-parameters"

In [0]:
checks_on = True

# 1 Load Maternity Interpreter Cohort Table

In [0]:

from pyspark.sql.window import Window
w2 = Window.partitionBy("uniqpregid").orderBy(f.col("est_preg_start"))

maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_cohort_gp_interaction_counts')

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (
    maternity_interpreter_cohort
    .select('person_id_mother_deid', 'uniqpregid', 'est_preg_start', 'lookback_start')
    # the below two lines are temporary - should be removed once its one record per pregnancy upstream
    .withColumn("row", f.row_number().over(w2))
    .filter(f.col("row") == 1).drop("row") 
)




In [0]:
if checks_on:
    count_var(maternity_interpreter_cohort, 'person_id_mother_deid')
    count_var(maternity_interpreter_cohort, 'uniqpregid')
    display(maternity_interpreter_cohort.limit(10))
    tab(maternity_interpreter_cohort, 'est_preg_start')
    tab(maternity_interpreter_cohort, 'lookback_start')

# 2 Load A+E healthcare use


In [0]:
from pyspark.sql import functions as F

#latest_date = str(spark.table(F'{dbc_old}.hes_ae_all_years_archive').select(F.max(F.col('archived_on') )).collect()[0][0])

#spark.table(F'{dbc_old}.hes_ae_all_years_archive').printSchema()
#print(F'Latest batch available: {latest_date}')
#print(F'Current batch being read: {tmp_archived_on}')

In [0]:

HES_AE= (spark.table(f'{dbc_old}.hes_ae_all_years_archive')
          .filter(F.col('archived_on') == tmp_archived_on)
            .select('archived_on', 'PERSON_ID_DEID', 'ARRIVALDATE', 'AEATTEND_EXC_PLANNED', 'DIAG3_01', 'DIAG2_01')
         .filter(F.col('ARRIVALDATE').isNotNull())
            .filter(F.col('ARRIVALDATE') >= "2018-01-01") # should change this to dynamic (use a param)
        .filter(F.col('ARRIVALDATE') <= "2024-12-31") # should change this to dynamic (use a param)
          #.filter(F.col)
          )


In [0]:
if checks_on:
    display(HES_AE.limit(5))
    print(HES_AE.select(F.min(F.col('ARRIVALDATE') )).collect()[0][0])
    print(HES_AE.select(F.max(F.col('ARRIVALDATE') )).collect()[0][0])
    print(HES_AE.count()) 


#3 Join HES A+E to Maternity interpreter cohort table 

In [0]:
##Attempting left join based on https://www.geeksforgeeks.org/pyspark-join-types-join-two-dataframes/

##check idcount in each table before and after join (idcount in output table will be same as left table idcount - filtered lookup)

# left join on two dataframes 
maternity_interpreter_HES_AE= (HES_AE
                              .join( F.broadcast(maternity_interpreter_cohort), HES_AE.PERSON_ID_DEID == maternity_interpreter_cohort.person_id_mother_deid,  "inner")
                              .filter(F.col('ARRIVALDATE') > F.col('lookback_start'))
                              .filter(F.col('ARRIVALDATE') < F.col('est_preg_start'))
)


Databricks data profile. Run in Databricks to view.

In [0]:
if checks_on:
    display(maternity_interpreter_HES_AE.sort("person_id_mother_deid", "ARRIVALDATE"), limit=10)
    count_var(maternity_interpreter_HES_AE, 'person_id_mother_deid')
    #count_var(maternity_interpreter_HES_AE, 'uniqpregid')
    #maternity_interpreter_HES_AE.printSchema()
    print(maternity_interpreter_HES_AE.select(F.min(F.col('ARRIVALDATE') )).collect()[0][0])
    print(maternity_interpreter_HES_AE.select(F.max(F.col('ARRIVALDATE') )).collect()[0][0])
    print(maternity_interpreter_HES_AE.select(F.min(F.col('lookback_start') )).collect()[0][0])
    print(maternity_interpreter_HES_AE.select(F.max(F.col('est_preg_start') )).collect()[0][0])
    #print(maternity_interpreter_HES_AE.count()) 

In [0]:
outName = f'{proj}_HES_AE_interaction_details'

# save
maternity_interpreter_HES_AE.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

#4. Collapse to one row per date for A+E attendances

In [0]:
w2 = Window.partitionBy("person_id_mother_deid").orderBy("person_id_mother_deid")

maternity_interpreter_HES_AE0 = (
    maternity_interpreter_HES_AE
    .select('person_id_mother_deid', 'uniqpregid', 'ARRIVALDATE')
    .dropDuplicates() #in order to separate to one visit per day
    .withColumn("row", f.row_number().over(w2))
    .withColumn("number_of_ae_interaction_days", f.max(f.col("row")).over(w2))
    .filter(f.col("row") == f.col("number_of_ae_interaction_days"))
    .drop("row") 
)

In [0]:
if checks_on:
    #maternity_interpreter_HES_AE0.printSchema()
    count_var( maternity_interpreter_HES_AE0, 'person_id_mother_deid')
    #maternity_interpreter_HES_AE0.count()
    display(maternity_interpreter_HES_AE0.orderBy("number_of_ae_interaction_days", ascending=False).limit(100))
    tab(maternity_interpreter_HES_AE0, "number_of_ae_interaction_days")

In [0]:
tab(maternity_interpreter_HES_AE0, "number_of_ae_interaction_days")

#5. Link cohort back to maternity interpreter cohort 
This is just so we have all the individuals who did not attend A+E in year preconception in the table again!

##5b. Load maternity interpreter cohort again

In [0]:
maternity_interpreter_cohort = spark.table(f'{dbc}.{proj}_cohort_gp_interaction_counts')
from pyspark.sql.window import Window
w2 = Window.partitionBy("uniqpregid").orderBy(f.col("est_preg_start"))

maternity_interpreter_cohort.printSchema()
maternity_interpreter_cohort = (
        maternity_interpreter_cohort
        # the below three lines are temporary - should be removed once its one record per pregnancy upstream
        .withColumn("row", f.row_number().over(w2))
        .filter(f.col("row") == 1)
        .drop("row") 
        .select('person_id_mother_deid', 'uniqpregid', 'est_preg_start',  'lookback_start', 'lookback_issue_flag', 'NHS_NUMBER_interpreter', 'SNOMED_conceptId', 'SNOMED_conceptId_description', 'DATE_interpreter', 'RECORD_DATE_interpreter', 'person_id_demo', 'Dob', 'eth5', 'region', 'imd_quintile', 'imd_decile', 'in_gdppr', 'gdppr_min_date', 'interpreter_use', 'record_before_lookback', 'ageatbookingmother', 'delivery_date', 'agefinal', 'folicacid', 'ovsvischcat', 'ovsvischcatappdate', 'complexsocialfactors', 'gestagebooking', 'gestagebookingweeks', 'previouslivebirths', 'previousstillbirths', 'previouslosseslessthan24weeks', 'parity', 'nulliparous', 'prev_loss', 'prev_preg', 'booking_after_10weeks', 'number_of_gp_interaction_days', 'fact_of_gp_interaction' )
        )

In [0]:
count_var(maternity_interpreter_cohort, 'person_id_mother_deid')
count_var(maternity_interpreter_cohort, 'uniqpregid')

# 6. Join HES_AE MSDS cohort to original interpreter cohort again


In [0]:
maternity_interpreter_HES_AE1 = (maternity_interpreter_cohort
       .join(maternity_interpreter_HES_AE0, on = ['person_id_mother_deid', 'uniqpregid' ]    , how = 'left' ) 
       .withColumn('number_of_ae_interaction_days', f.when(f.isnull('number_of_ae_interaction_days'), f.lit(0)).otherwise(f.col('number_of_ae_interaction_days')) )
       .withColumn('fact_of_ae_interaction', f.when(f.col('number_of_ae_interaction_days') > 0, f.lit('One_or_more')).otherwise(f.lit('No_interactions')) )
       #.where( f.col('ageatbookingmother') >= 18) # temporary - please apply this filter further upstream in pipeline - also consider upper bound age as well - appears to be a long tail in the data
)

In [0]:
outName = f'{proj}_cohort_ae_interaction_counts'

# save
maternity_interpreter_HES_AE1.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')

In [0]:
##check
if checks_on:
    display(maternity_interpreter_HES_AE1.limit(5))
    #count_var(maternity_interpreter_HES_AE1, 'uniqpregid')
    #maternity_interpreter_HES_AE1.printSchema()

In [0]:
#Load table from database
#maternity_interpreter_HES_AE1 = spark.table(f'{dbc}.{proj}_cohort_ae_interaction_counts')

#7. Work out proportion of patients who attended A+E in year preconception

In [0]:
tab(maternity_interpreter_HES_AE1, 'fact_of_ae_interaction')
tab(maternity_interpreter_HES_AE1, 'number_of_ae_interaction_days')

In [0]:
tab(maternity_interpreter_HES_AE1, 'fact_of_ae_interaction', 'interpreter_use')

# 8. Covid breakdown

## Filter to pre-COVID, spanning COVID or post-COVID

In [0]:
maternity_interpreter_HES_AEX = (
    maternity_interpreter_HES_AE1
    .withColumn('COVID_period_applicable', f.when(f.col('est_preg_start') < '2020-03-01', f.lit('Conception_before_Covid_start'))
                .when(f.col('lookback_start') >= '2020-03-01', f.lit('Lookback_period_after_Covid_start'))
                .otherwise(f.lit('Lookback_period_spans_Covid_start')) )
    )

In [0]:
display(maternity_interpreter_HES_AEX.select("COVID_period_applicable", "person_id_mother_deid", "lookback_start", "est_preg_start",  "interpreter_use").limit(100))

In [0]:
tab(maternity_interpreter_HES_AEX, 'COVID_period_applicable')

In [0]:
display(tab(maternity_interpreter_HES_AEX, 'COVID_period_applicable', 'fact_of_ae_interaction'))

##Interpreter users' A+E attendance by COVID

In [0]:
maternity_interpreter_HES_AEX2 = (maternity_interpreter_HES_AEX
                    .where( f.col('interpreter_use') == "yes")

)

In [0]:
display(tab(maternity_interpreter_HES_AEX2, 'COVID_period_applicable', 'fact_of_ae_interaction'))

##Non interpreter users A+E attendance by COVID

In [0]:
maternity_interpreter_HES_AEX3 = (maternity_interpreter_HES_AEX
                    .where( f.col('interpreter_use') == "no")

)

In [0]:
display(tab(maternity_interpreter_HES_AEX3, 'COVID_period_applicable', 'fact_of_ae_interaction'))

#9. Savepoint for COVID period included

In [0]:
outName = f'{proj}_cohort_ae_interaction_counts2'

# save
maternity_interpreter_HES_AEX.write.mode('overwrite').option("overwriteSchema", "true").saveAsTable(f'{dbc}.{outName}')